# 02 — Train the v1 baseline

Runs a short training job sized to fit inside a single Colab free-tier session (T4 GPU, ~12h, can disconnect without warning). Checkpoints save to a mounted Google Drive folder every epoch, so progress survives a disconnect even though this run itself is intentionally small — see `../README.md` for why this is a v1 baseline, not a fully converged model.

Run `01_explore_resplan.ipynb` first at least once, so you know the data pipeline works before spending GPU time.

In [ ]:
GITHUB_REPO_URL = ""  # e.g. "https://github.com/<you>/planHouse.git"
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/planHouse-ml/floor-plan-gen"  # used if GITHUB_REPO_URL is empty

import os
import sys

from google.colab import drive
drive.mount("/content/drive")  # needed either way, for checkpoint persistence

if GITHUB_REPO_URL:
    !git clone {GITHUB_REPO_URL} /content/planHouse
    PROJECT_DIR = "/content/planHouse/ml/floor-plan-gen"
else:
    PROJECT_DIR = DRIVE_PROJECT_PATH
    assert os.path.isdir(PROJECT_DIR), (
        f"{PROJECT_DIR} not found — upload ml/floor-plan-gen there first, "
        f"or set GITHUB_REPO_URL above instead."
    )

sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))
%pip install -q -r {os.path.join(PROJECT_DIR, "requirements.txt")}

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
from pathlib import Path

from floorplan_gen.data.download import download_resplan

DATA_DIR = Path(PROJECT_DIR) / "data"
download_resplan(DATA_DIR)  # no-op if already downloaded

## Train

`max_train_samples` caps the run so it comfortably fits in one session — raise it (or drop it entirely) once you have a sense of how long an epoch actually takes on the free T4. `checkpoint_dir` lives on Drive, not the Colab VM's local disk, specifically so a disconnect doesn't lose the run.

In [ ]:
from floorplan_gen.train import train

CHECKPOINT_DIR = Path("/content/drive/MyDrive/planHouse-ml/floor-plan-gen/checkpoints")

model = train(
    data_dir=DATA_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
    epochs=10,
    batch_size=32,
    lr=1e-3,
    overlap_weight=0.1,
    max_train_samples=2000,
)

## Next

Check that `train_loss`/`val_loss` printed above are actually decreasing epoch over epoch — if they aren't, don't move on yet (that's a real bug, not something `generate.py`'s geometry cleanup can paper over). Once they look reasonable, open `03_generate_demo.ipynb` and point it at `CHECKPOINT_DIR / "best.pt"`.